# Notebook 06 – Power BI Dashboard Preview

**Purpose.** Document and preview the Power BI dashboard that reads directly from the CSVs produced by earlier notebooks. 

Nothing in this notebook produces new forecasts or new decisions — it consumes the saved files.

**Source files**
| File | Produced by | Purpose |
|---|---|---|
| `pharma_sales_clean.csv` | Notebook 01 | Historical medicine sales |
| `blood_demand_clean.csv` | Notebook 01 | Historical blood demand and drivers |
| `final_forecasts.csv` | **Notebook 05** | Forecast values per item, design, and date |
| `stock_decisions.csv` | **Notebook 05** | Safety stock, reorder point, order quantity |
| `evaluation_summary.csv` | **Notebook 04** | Model quality metrics and intermittent flags |

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path(r"C:\Users\Administrator\MediShelf")
PROCESSED   = PROJECT_DIR / "data" / "processed"
PREVIEW_DIR = PROCESSED / "dashboard_preview"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

PHARMA_PATH = PROCESSED / "pharma_sales_clean.csv"
BLOOD_PATH  = PROCESSED / "blood_demand_clean.csv"
FC_PATH     = PROCESSED / "final_forecasts.csv"
STOCK_PATH  = PROCESSED / "stock_decisions.csv"
EVAL_PATH   = PROCESSED / "evaluation_summary.csv"

for p in (PHARMA_PATH, BLOOD_PATH, FC_PATH, STOCK_PATH, EVAL_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {p}. Did you run 04 and 05?")

def load_all():
    pharma = pd.read_csv(PHARMA_PATH)
    if "datum" in pharma.columns:
        pharma = pharma.rename(columns={"datum": "Date"})
    pharma["Date"] = pd.to_datetime(pharma["Date"])

    blood = pd.read_csv(BLOOD_PATH)
    blood["Date"] = pd.to_datetime(blood["Date"])

    fc = pd.read_csv(FC_PATH)
    fc["ForecastDate"] = pd.to_datetime(fc["ForecastDate"])

    stock = pd.read_csv(STOCK_PATH)
    evals = pd.read_csv(EVAL_PATH)
    return pharma, blood, fc, stock, evals

pharma, blood, fc, stock, evals = load_all()
print(f"Historical pharma rows : {len(pharma)}")
print(f"Historical blood rows  : {len(blood)}")
print(f"Forecast rows          : {len(fc)}")
print(f"Stock decision rows    : {len(stock)}")

Historical pharma rows : 2106
Historical blood rows  : 1827
Forecast rows          : 400
Stock decision rows    : 9


## Page 1 — KPI Overview

Python preview of the KPI cards that will appear in Power BI. In the `.pbix` these are **Card** visuals driven by the DAX measures listed at the end of this notebook.

In [4]:
def page1_kpis(stock, fc):
    n_items = stock.Item.nunique()
    total_ss = stock.Safety_Stock.sum()
    total_rop = stock.Reorder_Point.sum()
    avg_mase = fc.groupby("Item")["MASE"].first().mean()

    fig, axes = plt.subplots(1, 4, figsize=(14, 2.4))
    cards = [
        ("Items tracked", f"{n_items}"),
        ("Total safety stock (units)", f"{total_ss:,.0f}"),
        ("Total reorder point (units)", f"{total_rop:,.0f}"),
        ("Mean winner MASE", f"{avg_mase:.3f}"),
    ]
    for ax, (label, value) in zip(axes, cards):
        ax.text(0.5, 0.55, value, ha="center", va="center",
                fontsize=22, fontweight="bold")
        ax.text(0.5, 0.15, label, ha="center", va="center", fontsize=10)
        ax.axis("off")
    plt.suptitle("Page 1 — KPI overview (Power BI card preview)")
    plt.tight_layout()
    out = PREVIEW_DIR / "page1_kpis.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close()
    print("Wrote:", out)

page1_kpis(stock, fc)

Wrote: C:\Users\Administrator\MediShelf\data\processed\dashboard_preview\page1_kpis.png


## Page 2 — Blood Demand + Forecast + Stock Band

Shows historical weekly demand, the forecast, and the safety-stock band. In Power BI: a **Line chart** with the forecast series + a **Band** shaded range, and a **Table** of SS / ROP / OQ.

In [6]:
def page2_blood(blood, fc, stock):
    bw = blood.set_index("Date")["HistoricalBloodUsage"].resample("W-SUN").sum()
    bf = fc[fc.Item == "BLOOD"].sort_values("ForecastDate")
    bpol = stock[stock.Item == "BLOOD"].iloc[0]

    plt.figure(figsize=(12, 4.5))
    plt.plot(bw.index, bw.values, label="Historical weekly demand")
    if not bf.empty:
        plt.plot(bf.ForecastDate, bf.ForecastValue, color="orange",
                 label=f"Forecast ({bf.Model.iloc[0]})")
    plt.axhline(bpol.Reorder_Point, color="red", linestyle="--",
                label=f"Reorder point = {bpol.Reorder_Point:.0f}")
    plt.axhline(bpol.Safety_Stock, color="gray", linestyle=":",
                label=f"Safety stock = {bpol.Safety_Stock:.0f}")
    plt.title("Page 2 — Blood demand, forecast, and stock band")
    plt.ylabel("Units / week")
    plt.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    out = PREVIEW_DIR / "page2_blood.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close()
    print("Wrote:", out)

page2_blood(blood, fc, stock)

Wrote: C:\Users\Administrator\MediShelf\data\processed\dashboard_preview\page2_blood.png


## Page 2 — Blood Demand + Forecast + Stock Band

Shows historical weekly demand, the forecast, and the safety-stock band. In Power BI: a **Line chart** with the forecast series + a **Band** shaded range, and a **Table** of SS / ROP / OQ.

In [7]:
def page2_blood(blood, fc, stock):
    bw = blood.set_index("Date")["HistoricalBloodUsage"].resample("W-SUN").sum()
    bf = fc[fc.Item == "BLOOD"].sort_values("ForecastDate")
    bpol = stock[stock.Item == "BLOOD"].iloc[0]

    plt.figure(figsize=(12, 4.5))
    plt.plot(bw.index, bw.values, label="Historical weekly demand")
    if not bf.empty:
        plt.plot(bf.ForecastDate, bf.ForecastValue, color="orange",
                 label=f"Forecast ({bf.Model.iloc[0]})")
    plt.axhline(bpol.Reorder_Point, color="red", linestyle="--",
                label=f"Reorder point = {bpol.Reorder_Point:.0f}")
    plt.axhline(bpol.Safety_Stock, color="gray", linestyle=":",
                label=f"Safety stock = {bpol.Safety_Stock:.0f}")
    plt.title("Page 2 — Blood demand, forecast, and stock band")
    plt.ylabel("Units / week")
    plt.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    out = PREVIEW_DIR / "page2_blood.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close()
    print("Wrote:", out)

page2_blood(blood, fc, stock)

Wrote: C:\Users\Administrator\MediShelf\data\processed\dashboard_preview\page2_blood.png


## Page 3 — Medicine Demand per Group

One small multiple per ATC group: history + forecast. In Power BI use a **Line chart** per page or a small-multiples visual with `ATC_Group` as the facet.

In [8]:
def page3_medicine(pharma, fc):
    pdf = pharma.set_index("Date")
    groups = [g for g in fc.Item.unique() if g != "BLOOD"]
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, g in zip(axes.ravel(), groups):
        hist = pdf[g].resample("W-SUN").sum()
        sub = fc[(fc.Item == g) & (fc.Design == "Weekly")]
        if sub.empty:
            sub = fc[(fc.Item == g) & (fc.Design == "Monthly")]
        ax.plot(hist.index, hist.values, linewidth=0.8)
        if not sub.empty:
            ax.plot(sub.ForecastDate, sub.ForecastValue, color="orange")
        ax.set_title(g, fontsize=10)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)
    plt.suptitle("Page 3 — Historical vs forecast per ATC group")
    plt.tight_layout()
    out = PREVIEW_DIR / "page3_medicine.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close()
    print("Wrote:", out)

page3_medicine(pharma, fc)

Wrote: C:\Users\Administrator\MediShelf\data\processed\dashboard_preview\page3_medicine.png


## Page 4 — Stock Recommendations Table

In Power BI: **Table** visual bound to `stock_decisions.csv` with conditional background colours (Safety stock gradient, red ROP when breached, etc.).

In [9]:
def page4_stock_table(stock):
    fig, ax = plt.subplots(figsize=(12, 0.6 + 0.35 * len(stock)))
    ax.axis("off")
    cells = [stock.columns.tolist()] + stock.astype(str).values.tolist()
    tbl = ax.table(cellText=cells, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.3)
    for j in range(len(stock.columns)):
        tbl[0, j].set_facecolor("#444")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    plt.title("Page 4 — Stock recommendations per item")
    plt.tight_layout()
    out = PREVIEW_DIR / "page4_stock_table.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close()
    print("Wrote:", out)

page4_stock_table(stock)

Wrote: C:\Users\Administrator\MediShelf\data\processed\dashboard_preview\page4_stock_table.png


## Page 5 — Model Quality (MASE + MAPE + intermittent flag)

MASE is the primary quality indicator. MAPE is shown but marked with the intermittent flag from Notebook 04; when `Intermittent = TRUE`, MAPE should be visually de-emphasized because zeros distort it.

In [10]:
def page5_model_quality(fc, evals):
    q = (
        fc.groupby(["Item", "Design", "Model"])
        .agg(MASE=("MASE", "first"))
        .reset_index()
    )
    flag = (
        evals[["Group", "Design", "Model", "Intermittent", "Avg_MAPE"]]
        .rename(columns={"Group": "Item"})
    )
    q = q.merge(flag, on=["Item", "Design", "Model"], how="left")
    q["MAPE_warning"] = np.where(
        q.Intermittent.eq(True),
        "MAPE unreliable (intermittent demand)", "",
    )

    fig, ax = plt.subplots(figsize=(12, 0.6 + 0.35 * len(q)))
    ax.axis("off")
    cells = [q.columns.tolist()] + q.round(3).astype(str).values.tolist()
    tbl = ax.table(cellText=cells, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.3)
    for j in range(len(q.columns)):
        tbl[0, j].set_facecolor("#444")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    plt.title("Page 5 — Model quality (MASE primary; MAPE flagged when unreliable)")
    plt.tight_layout()
    out = PREVIEW_DIR / "page5_model_quality.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close()
    print("Wrote:", out)

page5_model_quality(fc, evals)

Wrote: C:\Users\Administrator\MediShelf\data\processed\dashboard_preview\page5_model_quality.png


## DAX Recipes & Power Query Steps

In [11]:
DAX_RECIPES = """
========================================================================
DAX recipes to paste into the Power BI .pbix
========================================================================
Total Safety Stock         = SUM(stock_decisions[Safety_Stock])
Total Reorder Point        = SUM(stock_decisions[Reorder_Point])
Mean Winner MASE           = AVERAGE(final_forecasts[MASE])
Items Tracked              = DISTINCTCOUNT(stock_decisions[Item])

Forecast Band Upper        = final_forecasts[ForecastValue]
                             + 1.65 * MAX(stock_decisions[Sigma_Weekly_Demand])
Forecast Band Lower        = MAX(0,
                               final_forecasts[ForecastValue]
                               - 1.65 * MAX(stock_decisions[Sigma_Weekly_Demand]))

Stock Status =
  VAR Current = [Total Safety Stock]
  VAR ROP     = [Total Reorder Point]
  RETURN IF(Current < ROP, "REORDER", "OK")

MAPE Warning =
  IF(SELECTEDVALUE(evaluation_summary[Intermittent]),
     "MAPE unreliable for intermittent demand",
     BLANK())

Power Query load steps (for each CSV):
  Get Data -> Text/CSV -> Transform Data
  -> Promote headers -> set date & numeric column types
  -> Mark Date table.
========================================================================
"""
print(DAX_RECIPES)

# --- Smoke checks
assert len(stock) >= 8
assert {"Safety_Stock", "Reorder_Point", "Suggested_Order_Quantity"} <= set(stock.columns)
assert fc.ForecastValue.notna().all()
assert (fc.ForecastValue >= 0).all()
print("\nVerification passed — dashboard data is complete.")


DAX recipes to paste into the Power BI .pbix
Total Safety Stock         = SUM(stock_decisions[Safety_Stock])
Total Reorder Point        = SUM(stock_decisions[Reorder_Point])
Mean Winner MASE           = AVERAGE(final_forecasts[MASE])
Items Tracked              = DISTINCTCOUNT(stock_decisions[Item])

Forecast Band Upper        = final_forecasts[ForecastValue]
                             + 1.65 * MAX(stock_decisions[Sigma_Weekly_Demand])
Forecast Band Lower        = MAX(0,
                               final_forecasts[ForecastValue]
                               - 1.65 * MAX(stock_decisions[Sigma_Weekly_Demand]))

Stock Status =
  VAR Current = [Total Safety Stock]
  VAR ROP     = [Total Reorder Point]
  RETURN IF(Current < ROP, "REORDER", "OK")

MAPE Warning =
  IF(SELECTEDVALUE(evaluation_summary[Intermittent]),
     "MAPE unreliable for intermittent demand",
     BLANK())

Power Query load steps (for each CSV):
  Get Data -> Text/CSV -> Transform Data
  -> Promote headers -> set d